In [3]:
import os

def collect_parent_ids(subset):
    base_dataset = getattr(subset, "dataset", subset)
    indices = getattr(subset, "indices", range(len(subset)))

    parent_ids = set()
    for idx in indices:
        filepath = base_dataset.samples[idx][0]
        filename = os.path.basename(filepath)
        
        # 1. Clean the prefixes
        if filename.startswith("transfer_"):
            filename = filename[len("transfer_"):]
        elif filename.startswith("redigital_"):
            filename = filename[len("redigital_"):]
            
        # 2. Strip the extension (.png or .jpg)
        parent_id, _ = os.path.splitext(filename)
        parent_ids.add(parent_id)
        
    return parent_ids

# Create the loaders if not already in memory
if 'train_loader' not in globals() or 'val_loader' not in globals() or 'test_loader' not in globals():
    from src.data.dataloader import get_dataloaders
    train_loader, val_loader, test_loader = get_dataloaders(data_dir="src/data/RRDataset_final", batch_size=64)

# Get subsets from loaders
train_subset = train_loader.dataset
val_subset = val_loader.dataset

print("Scanning Training Set (Extracting parent IDs)...")
train_parent_ids = collect_parent_ids(train_subset)

print("Scanning Validation Set (Extracting parent IDs)...")
val_parent_ids = collect_parent_ids(val_subset)

# Find the true leakage
leaked_parents = train_parent_ids.intersection(val_parent_ids)

print("\n--- TRUE LEAKAGE REPORT ---")
print(f"Unique parent images in Train: {len(train_parent_ids)}")
print(f"Unique parent images in Val:   {len(val_parent_ids)}")
print(f"TRUE LEAKED PARENT IMAGES:     {len(leaked_parents)}")

if len(leaked_parents) > 0:
    print(f"Percentage of Val subjects leaked: {(len(leaked_parents)/len(val_parent_ids))*100:.2f}%")
    print("\nExamples of leaked parent subjects:")
    for parent in list(leaked_parents)[:5]:
        print(f" - {parent}")
else:
    print("\nZero leakage! Your split is truly safe.")


Loaded 12 images.
Scanning Training Set (Extracting parent IDs)...
Scanning Validation Set (Extracting parent IDs)...

--- TRUE LEAKAGE REPORT ---
Unique parent images in Train: 9
Unique parent images in Val:   1
TRUE LEAKED PARENT IMAGES:     0

Zero leakage! Your split is truly safe.


In [4]:
from src.data.dataloader import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(data_dir="src/data/RRDataset_final", batch_size=64)

# Grab one batch
sample = next(iter(train_loader))

print(f"Images shape: {sample[0].shape}")       # Should be [32, 3, 224, 224]
print(f"Real/Fake labels: {sample[1]}")      # Should be a mix of 0s and 1s
print(f"Transform labels: {sample[2]}")   # Should be a mix of 0s, 1s, and 2s

Loaded 12 images.


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Images shape: torch.Size([9, 3, 224, 224])
Real/Fake labels: tensor([1, 0, 1, 1, 0, 0, 1, 1, 0])
Transform labels: tensor([1, 2, 0, 2, 1, 1, 0, 1, 0])


In [6]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Import your custom modules directly from the src directory
from src.train.loss import MultiTaskLoss
from src.train.loops import MultiTaskModel, train_epoch
from src.train.ablation import run_ablation_study

# Set device to GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
    )
print(f"Using device: {device}")

Using device: cpu


## ResNet-50 Flow

Train ResNet-50 across the full alpha/beta weighting sweep plus the learned-uncertainty variant, then evaluate every checkpoint on the held-out test set and visualize accuracy trade-offs and GradCAM attention.

In [ ]:
print("Starting ResNet-50 alpha/beta sweep...")
run_ablation_study(train_loader, val_loader, backbone_type='resnet50', loss_type='fixed')

print("\nStarting ResNet-50 uncertainty-weighted run...")
run_ablation_study(train_loader, val_loader, backbone_type='resnet50', loss_type='uncertainty')

In [ ]:
import os
import torch
from src.train.loops import MultiTaskModel
from src.evaluation.visualizer import evaluate_model, plot_ablation_study

resnet_checkpoints = {
    '1.0_0.0': 'models/model_10_00.pth',
    '0.0_1.0': 'models/model_00_10.pth',
    '0.5_0.5': 'models/model_05_05.pth',
    '0.8_0.2': 'models/model_08_02.pth',
    '0.2_0.8': 'models/model_02_08.pth',
    'uncertainty': 'models/model_uncertainty.pth'
}

resnet_ablation_results = {}

for weight_key, file_path in resnet_checkpoints.items():
    if os.path.exists(file_path):
        print(f"Evaluating ResNet-50 [{weight_key}]...")
        model = MultiTaskModel(backbone_type='resnet50').to(device)
        model.load_state_dict(torch.load(file_path, map_location=device))
        acc_rf, acc_trans = evaluate_model(model, test_loader, device, quiet_mode=True)
        resnet_ablation_results[weight_key] = (acc_rf * 100, acc_trans * 100)

print("\nDetailed evaluation for ResNet-50 [0.5/0.5]...")
detailed_model = MultiTaskModel(backbone_type='resnet50').to(device)
detailed_model.load_state_dict(torch.load('models/model_05_05.pth', map_location=device))
evaluate_model(detailed_model, test_loader, device, dataset_name="Test")

plot_ablation_study(resnet_ablation_results, save_path='ablation_study_resnet50.png')

In [ ]:
# --- GradCAM Visualization for ResNet-50 ---
from src.evaluation.visualizer import plot_gradcam
from src.train.loops import MultiTaskModel
resnet_model = MultiTaskModel(backbone_type='resnet50').to(device)
resnet_model.load_state_dict(torch.load('models/model_05_05.pth', map_location=device))
plot_gradcam(resnet_model, val_loader, device, num_samples=4, save_path='gradcam_resnet50.png')

## ConvNeXt-Tiny Flow

Train ConvNeXt-Tiny across the full alpha/beta weighting sweep plus the learned-uncertainty variant, then evaluate every checkpoint on the held-out test set and visualize accuracy trade-offs and GradCAM attention.

In [ ]:
print("Starting ConvNeXt-Tiny alpha/beta sweep...")
run_ablation_study(train_loader, val_loader, backbone_type='convnext_tiny', loss_type='fixed')

print("\nStarting ConvNeXt-Tiny uncertainty-weighted run...")
run_ablation_study(train_loader, val_loader, backbone_type='convnext_tiny', loss_type='uncertainty')

In [ ]:
import os
import torch
from src.train.loops import MultiTaskModel
from src.evaluation.visualizer import evaluate_model, plot_ablation_study

convnext_checkpoints = {
    '1.0_0.0': 'models/model_convnext_tiny_10_00.pth',
    '0.0_1.0': 'models/model_convnext_tiny_00_10.pth',
    '0.5_0.5': 'models/model_convnext_tiny_05_05.pth',
    '0.8_0.2': 'models/model_convnext_tiny_08_02.pth',
    '0.2_0.8': 'models/model_convnext_tiny_02_08.pth',
    'uncertainty': 'models/model_convnext_tiny_uncertainty.pth'
}

convnext_ablation_results = {}

for weight_key, file_path in convnext_checkpoints.items():
    if os.path.exists(file_path):
        print(f"Evaluating ConvNeXt-Tiny [{weight_key}]...")
        model = MultiTaskModel(backbone_type='convnext_tiny').to(device)
        model.load_state_dict(torch.load(file_path, map_location=device))
        acc_rf, acc_trans = evaluate_model(model, test_loader, device, quiet_mode=True)
        convnext_ablation_results[weight_key] = (acc_rf * 100, acc_trans * 100)

print("\nDetailed evaluation for ConvNeXt-Tiny [0.5/0.5]...")
detailed_model = MultiTaskModel(backbone_type='convnext_tiny').to(device)
detailed_model.load_state_dict(torch.load('models/model_convnext_tiny_05_05.pth', map_location=device))
evaluate_model(detailed_model, test_loader, device, dataset_name="Test")

plot_ablation_study(convnext_ablation_results, save_path='ablation_study_convnext_tiny.png')

In [ ]:
# --- GradCAM Visualization for ConvNeXt-Tiny ---
from src.evaluation.visualizer import plot_gradcam
from src.train.loops import MultiTaskModel
convnext_model = MultiTaskModel(backbone_type='convnext_tiny').to(device)
convnext_model.load_state_dict(torch.load('models/model_convnext_tiny_05_05.pth', map_location=device))
plot_gradcam(convnext_model, val_loader, device, num_samples=4, save_path='gradcam_convnext_tiny.png')

## Cross-Backbone Comparison

Compare the best ResNet-50 and ConvNeXt-Tiny checkpoints head-to-head on the test set.

In [ ]:
# --- Backbone Architecture Comparison ---
from src.evaluation.compare_backbones import compare_backbone_checkpoints
models_to_compare = {
    "ResNet-50": ("resnet50", "models/model_05_05.pth"),
    "ConvNeXt-Tiny": ("convnext_tiny", "models/model_convnext_tiny_05_05.pth"),
}
comparison_results = compare_backbone_checkpoints(models_to_compare, test_loader=test_loader)